In [1]:
# Load environment and suppress warnings

import os
import warnings
from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings("ignore")

api_key = os.getenv("OPENAI_API_KEY")
print("Key loaded:", bool(api_key))

if not api_key:
    raise ValueError("OPENAI_API_KEY not found.")

Key loaded: True


In [2]:
# Import LangChain components and initialize deterministic model
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

if os.getenv("OPENAI_API_KEY"):
    model = ChatOpenAI(model="gpt-5-mini", temperature=0)
else:
    model = None
    print("Skipping model init. No API key found.")


In [3]:
# Step 1: Summarize ticket

summary_prompt = PromptTemplate.from_template(
"""
Summarize the support ticket in one concise sentence.

Ticket:
{ticket_text}
"""
)

summary_chain = summary_prompt | model | StrOutputParser()

In [4]:
# Step 2: Priority classification (buggy - not enforcing strict label)

priority_prompt = PromptTemplate.from_template(
"""
Explain the urgency level of this ticket.

Summary:
{summary}
"""
)

priority_chain = priority_prompt | model | StrOutputParser()

In [5]:
# Step 3: Action depends on exact priority label

action_prompt = PromptTemplate.from_template(
"""
Follow these rules:

If priority is High -> escalate immediately.
If priority is Medium -> assign to support queue.
If priority is Low -> respond within 48 hours.

Return only the action statement.

Summary:
{summary}

Priority:
{priority}
"""
)

action_chain = action_prompt | model | StrOutputParser()

In [6]:
# Execute workflow and print intermediate outputs

ticket_text = "Customer reports complete system outage affecting all users."

summary = summary_chain.invoke({"ticket_text": ticket_text})
print("=== STEP 1: SUMMARY ===")
print(summary)
print()

priority = priority_chain.invoke({"summary": summary})
print("=== STEP 2: PRIORITY (UNEXPECTED FORMAT) ===")
print(priority)
print()

action = action_chain.invoke({
    "summary": summary,
    "priority": priority
})

print("=== STEP 3: ACTION ===")
print(action)

=== STEP 1: SUMMARY ===
Customer reports a complete system outage affecting all users.

=== STEP 2: PRIORITY (UNEXPECTED FORMAT) ===
Urgency: Critical — immediate action required (Sev 1 / P0 / Production outage)

Why:
- The system is completely down and no users can use it — total service outage.
- Production impact across the entire user base means likely revenue, customer SLA, and reputational risk.
- This meets the usual criteria for the highest-severity incident: immediate paging/escalation and an all-hands response.

Recommended immediate actions (first 30–60 minutes):
1. Page/on-call: Immediately page the on-call incident response team, engineering, SRE/ops, and any required vendors.
2. Open an incident channel: Create a dedicated Slack/MS Teams/War Room and an incident ticket with clear ownership (Incident Commander).
3. Triage and data collection: Gather error logs, metrics/dashboards, recent deploy/change history, network status, and any outage patterns.
4. Mitigation/runbooks

In [7]:
# Inspect priority format

print("=== DEBUG CHECK ===")
print("Raw priority:", priority)
print("Is strict label:", priority.strip() in {"High", "Medium", "Low"})

=== DEBUG CHECK ===
Raw priority: Urgency: Critical — immediate action required (Sev 1 / P0 / Production outage)

Why:
- The system is completely down and no users can use it — total service outage.
- Production impact across the entire user base means likely revenue, customer SLA, and reputational risk.
- This meets the usual criteria for the highest-severity incident: immediate paging/escalation and an all-hands response.

Recommended immediate actions (first 30–60 minutes):
1. Page/on-call: Immediately page the on-call incident response team, engineering, SRE/ops, and any required vendors.
2. Open an incident channel: Create a dedicated Slack/MS Teams/War Room and an incident ticket with clear ownership (Incident Commander).
3. Triage and data collection: Gather error logs, metrics/dashboards, recent deploy/change history, network status, and any outage patterns.
4. Mitigation/runbooks: Execute hotfix/rollback/failover runbooks as appropriate; if no runbook, attempt safe rollback to

In [8]:
# Fix Step 2 by enforcing strict output

priority_prompt_fixed = PromptTemplate.from_template(
"""
Classify priority as exactly one of:
High
Medium
Low

Return only the priority label.

Summary:
{summary}
"""
)

priority_chain_fixed = priority_prompt_fixed | model | StrOutputParser()

In [9]:
# Re-run workflow with fixed transformation

priority_fixed = priority_chain_fixed.invoke({"summary": summary})

print("=== STEP 2: PRIORITY (FIXED) ===")
print(priority_fixed)
print()

action_fixed = action_chain.invoke({
    "summary": summary,
    "priority": priority_fixed
})

print("=== STEP 3: ACTION (FIXED) ===")
print(action_fixed)

=== STEP 2: PRIORITY (FIXED) ===
High

=== STEP 3: ACTION (FIXED) ===
Escalate immediately.
